In [1]:
from scripts.transformer_prediction_interface.base import DoPFNRegressor
import torch

In [2]:
import numpy as np
from copy import deepcopy


def estimate_ate_dopfn(X: np.ndarray, t: np.ndarray, y: np.ndarray) -> float:
    X_t_train = np.concatenate(
        [t[:, None], X],
        axis=1,
    )

    dopfn = DoPFNRegressor()
    dopfn.fit(X_t_train, y)

    x_1, x_0 = deepcopy(X), deepcopy(X)
    X_test_0 = np.concatenate(
        [
            np.zeros((x_0.shape[0], 1)),
            x_0,
        ],
        axis=1,
    )
    X_test_1 = np.concatenate(
        [
            np.ones((x_1.shape[0], 1)),
            x_1,
        ],
        axis=1,
    )

    y_test_0 = dopfn.predict(torch.from_numpy(X_test_0))
    y_test_1 = dopfn.predict(torch.from_numpy(X_test_1))
    return (y_test_1 - y_test_0).mean()


def estimate_cate_dopfn(X_train: np.ndarray, t_train: np.ndarray, y_train: np.ndarray, X_test: np.ndarray):
    X_t_train = np.concatenate(
        [t_train[:, None], X_train],
        axis=1,
    )

    dopfn = DoPFNRegressor()
    dopfn.fit(X_t_train, y_train)

    x_1, x_0 = deepcopy(X_test), deepcopy(X_test)
    X_test_0 = np.concatenate(
        [
            np.zeros((x_0.shape[0], 1)),
            x_0,
        ],
        axis=1,
    )
    X_test_1 = np.concatenate(
        [
            np.ones((x_1.shape[0], 1)),
            x_1,
        ],
        axis=1,
    )

    y_test_0 = dopfn.predict(torch.from_numpy(X_test_0))
    y_test_1 = dopfn.predict(torch.from_numpy(X_test_1))
    return y_test_1 - y_test_0

## Sales Dataset

In [3]:
from datasets import load_dataset

dataset = load_dataset(ds_name="sales")

train_ds, test_ds = dataset.generate_valid_split(n_splits=2)

cate_pred = estimate_cate_dopfn(
    X_train=train_ds.x_obs[:, 1:], t_train=train_ds.x_obs[:, 0], y_train=train_ds.y_obs, X_test=test_ds.x[:, 1:]
)

Semi-Real Benchmarks: 100%|██████████| 2/2 [00:00<00:00,  2.08it/s]


Changed model to be compatible with CPU, this is needed for the current version of PyTorch, see issue: https://github.com/pytorch/pytorch/issues/97128. The model will be slower if reused on GPU.


In [16]:
from copy import deepcopy
import pandas as pd
from dowhy import gcm
from dowhy.gcm.auto import AssignmentQuality

graph = train_ds.function_args["graph"]

graph_nodes = deepcopy(graph.nodes)
for node in graph_nodes:
    if node not in train_ds.attribute_names:
        graph.remove_node(node)

causal_model = gcm.InvertibleStructuralCausalModel(graph)

train_df = pd.DataFrame(torch.concat([train_ds.x, train_ds.y.unsqueeze(1)], axis=1), columns=train_ds.attribute_names)
test_df = pd.DataFrame(
    torch.concat([test_ds.x_obs, test_ds.y_obs.unsqueeze(1)], axis=1), columns=test_ds.attribute_names
)

gcm.auto.assign_causal_mechanisms(causal_model, train_df, AssignmentQuality.BETTER)
gcm.fit(causal_model, train_df)

samples_do_1 = gcm.counterfactual_samples(
    interventions={test_ds.do_scm.scm.t_key: lambda x: 1.0},
    causal_model=causal_model,
    observed_data=test_df,
)

samples_do_0 = gcm.counterfactual_samples(
    interventions={test_ds.do_scm.scm.t_key: lambda x: 0.0},
    causal_model=causal_model,
    observed_data=test_df,
)

cate_true = samples_do_1[test_ds.do_scm.scm.y_key].values - samples_do_0[test_ds.do_scm.scm.y_key].values

Fitting causal mechanism of node Operational Cost: 100%|██████████| 8/8 [00:00<00:00, 105.87it/s]


In [15]:
def n_mse(pred, true):
    return (((pred - true) / (true.max() - true.min())) ** 2).mean()


n_mse(cate_pred, cate_true)

## Other Benchmarks

In [17]:
from typing import Any

import numpy as np
import pandas as pd

from abc import ABC, abstractmethod
from dataclasses import dataclass

import numpy as np


@dataclass
class CATE_Dataset:  # conditional average treatment effect
    X_train: np.ndarray
    t_train: np.ndarray
    y_train: np.ndarray
    X_test: np.ndarray
    true_cate: np.ndarray


@dataclass
class ATE_Dataset:  # average treatment effect
    X: np.ndarray
    t: np.ndarray
    y: np.ndarray
    true_ate: float


class EvalDatasetCatalog(ABC):
    """
    The dataset catalog is a collection of datasets used for evaluating the model.
    """

    def __init__(self, n_tables: int, name: str):
        self.n_tables = n_tables
        self.name = name

    def __len__(self):
        return self.n_tables

    def __str__(self):
        return self.name

    @abstractmethod
    def __getitem__(self, index) -> Any:
        raise NotImplementedError("This method should be implemented by the subclass")

## ACIC 2016 Dataset

In [18]:
# The ACIC 2016 challenge dataset
#
# Sources:
# [1] Dorie, Vincent, et al. "Automated versus do-it-yourself methods for causal inference: Lessons learned
# from a data analysis competition." (2019): 43-68.
# [2] https://github.com/BiomedSciAI/causallib/tree/master/causallib/datasets/data/acic_challenge_2016
#
# The challenge includes 10 different datasets.

from typing import Tuple

X_CSV_URL = (
    "https://raw.githubusercontent.com/BiomedSciAI/causallib/master/causallib/datasets/data/acic_challenge_2016/x.csv"
)

ZY_CSV_URL = (
    lambda i: f"https://raw.githubusercontent.com/BiomedSciAI/causallib/master/causallib/datasets/data/acic_challenge_2016/zymu_{i}.csv"
)


class ACIC2016Dataset(EvalDatasetCatalog):
    def __init__(self, test_ratio: float = 0.1, seed: int = 42, n_tables: int = 10):
        super().__init__(n_tables, name="ACIC2016")
        self.test_ratio = test_ratio
        self.x_data = pd.read_csv(X_CSV_URL)
        self.rngs = [np.random.default_rng(seed + i) for i in range(n_tables)]
        self.datasets = [self._get_data(i) for i in range(n_tables)]

    def _get_data(self, idx: int) -> CATE_Dataset:
        """Loads and processes a single dataset split."""
        # Download file URLs
        simulation_url = ZY_CSV_URL(idx + 1)

        sim_data = pd.read_csv(simulation_url)

        # Define column names for x.csv and simulation data
        self.x_data.columns = [f"x_{i+1}" for i in range(self.x_data.shape[1])]
        sim_data.columns = ["z", "y0", "y1", "mu0", "mu1"]

        # Handle categorical variables
        categorical_columns = ["x_2", "x_21", "x_24"]
        numerical_columns = [f"x_{i+1}" for i in range(self.x_data.shape[1]) if f"x_{i+1}" not in categorical_columns]
        self.x_data["x_2_numeric"] = self.x_data["x_2"].astype("category").cat.codes
        self.x_data["x_21_numeric"] = self.x_data["x_21"].astype("category").cat.codes
        self.x_data["x_24_numeric"] = self.x_data["x_24"].astype("category").cat.codes
        numerical_columns = numerical_columns + ["x_2_numeric", "x_21_numeric", "x_24_numeric"]
        self.x_data = self.x_data.loc[:, numerical_columns]

        # Convert to tensors
        covariates = self.x_data.values.astype(np.float32)  # Covariates with encoded categorical variables
        treatments = sim_data["z"].values.astype(np.float32)  # Treatment

        y1 = sim_data["y1"].values.astype(np.float32)  # Potential outcomes under treatment
        y0 = sim_data["y0"].values.astype(np.float32)  # Potential outcomes under control
        outcomes = np.where(treatments == 1, y1, y0)

        mu0 = sim_data["mu0"].values.astype(np.float32)
        mu1 = sim_data["mu1"].values.astype(np.float32)
        cate = mu1 - mu0
        ate = cate.mean().item()

        # Split the dataset into train and test sets
        indices = self.rngs[idx].permutation(covariates.shape[0])
        split_idx = int(len(indices) * (1 - self.test_ratio))
        train_indices = indices[:split_idx]
        test_indices = indices[split_idx:]
        cate_dataset = CATE_Dataset(
            X_train=covariates[train_indices],
            t_train=treatments[train_indices],
            y_train=outcomes[train_indices],
            X_test=covariates[test_indices],
            true_cate=cate[test_indices],
        )
        ate_dataset = ATE_Dataset(
            X=covariates,
            t=treatments,
            y=outcomes,
            true_ate=ate,
        )

        return cate_dataset, ate_dataset

    def __getitem__(self, index) -> Tuple[CATE_Dataset, ATE_Dataset]:
        return self.datasets[index]

In [19]:
dataset = ACIC2016Dataset()

pehes = []
ate_rel_errs = []

for i in range(len(dataset)):
    cate_dset, ate_dset = dataset[i]
    cate_pred = estimate_cate_dopfn(
        X_train=cate_dset.X_train,
        t_train=cate_dset.t_train,
        y_train=cate_dset.y_train,
        X_test=cate_dset.X_test,
    )
    pehe = np.sqrt(np.mean((cate_pred - cate_dset.true_cate) ** 2))
    pehes.append(pehe)

    ate_pred = estimate_ate_dopfn(
        X=ate_dset.X,
        t=ate_dset.t,
        y=ate_dset.y,
    )
    ate_rel_err = np.abs(ate_pred - ate_dset.true_ate) / np.abs(ate_dset.true_ate)
    ate_rel_errs.append(ate_rel_err)

print(f"PEHE on ACIC datasets: {np.mean(pehes):.4f} ± {np.std(pehes) / np.sqrt(len(pehes)):.4f}")
print(
    f"ATE relative error on ACIC datasets: {np.mean(ate_rel_errs):.4f} ± {np.std(ate_rel_errs) / np.sqrt(len(ate_rel_errs)):.4f}"
)

PEHE on ACIC datasets: 4.1144 ± 0.5198
ATE relative error on ACIC datasets: 0.6666 ± 0.0389


## IHDP Dataset

In [20]:
import os

current_dir = os.getcwd()


class IHDPDataset(EvalDatasetCatalog):
    def __init__(
        self,
        n_tables: int = 100,
        seed: int = 42,
    ):
        n_tables = min(n_tables, 100)
        self.rngs = [np.random.default_rng(seed + i) for i in range(n_tables)]
        self.train_data = np.load(os.path.join(current_dir, "data/IHDP/ihdp_npci_1-100.train.npz"))
        self.test_data = np.load(os.path.join(current_dir, "data/IHDP/ihdp_npci_1-100.test.npz"))
        super().__init__(n_tables, name="IHDP")

    def __getitem__(self, idx: int) -> Tuple[CATE_Dataset, ATE_Dataset]:
        train_covariates = self.train_data["x"][..., idx].astype(np.float32)  # Covariates
        train_treatments = self.train_data["t"][..., idx].astype(np.float32)  # Treatment
        train_outcomes = self.train_data["yf"][..., idx].astype(np.float32)  # Outcomes

        test_covariates = self.test_data["x"][..., idx].astype(np.float32)  # Covariates
        test_treatments = self.test_data["t"][..., idx].astype(np.float32)  # Treatment
        test_outcomes = self.test_data["yf"][..., idx].astype(np.float32)
        test_mu1 = self.test_data["mu1"][..., idx].astype(np.float32)
        test_mu0 = self.test_data["mu0"][..., idx].astype(np.float32)

        test_cate = test_mu1 - test_mu0
        ate = float(self.train_data["ate"].item())

        # combine test and train and permute for ATE
        all_covariates = np.concatenate([train_covariates, test_covariates], axis=0)
        all_treatments = np.concatenate([train_treatments, test_treatments], axis=0)
        all_outcomes = np.concatenate([train_outcomes, test_outcomes], axis=0)
        indices = self.rngs[idx].permutation(all_covariates.shape[0])

        cate_dataset = CATE_Dataset(
            X_train=train_covariates,
            t_train=train_treatments,
            y_train=train_outcomes,
            X_test=test_covariates,
            true_cate=test_cate,
        )

        ate_dataset = ATE_Dataset(
            X=all_covariates[indices],
            t=all_treatments[indices],
            y=all_outcomes[indices],
            true_ate=ate,
        )

        return cate_dataset, ate_dataset

In [21]:
dataset = IHDPDataset()

pehes = []
ate_rel_errs = []

for i in range(len(dataset)):
    cate_dset, ate_dset = dataset[i]
    cate_pred = estimate_cate_dopfn(
        X_train=cate_dset.X_train,
        t_train=cate_dset.t_train,
        y_train=cate_dset.y_train,
        X_test=cate_dset.X_test,
    )
    pehe = np.sqrt(np.mean((cate_pred - cate_dset.true_cate) ** 2))
    pehes.append(pehe)

    ate_pred = estimate_ate_dopfn(
        X=ate_dset.X,
        t=ate_dset.t,
        y=ate_dset.y,
    )
    ate_rel_err = np.abs(ate_pred - ate_dset.true_ate) / np.abs(ate_dset.true_ate)
    ate_rel_errs.append(ate_rel_err)

print(f"PEHE on IHDP datasets: {np.mean(pehes):.4f} ± {np.std(pehes) / np.sqrt(len(pehes)):.4f}")
print(
    f"ATE relative error on IHDP datasets: {np.mean(ate_rel_errs):.4f} ± {np.std(ate_rel_errs) / np.sqrt(len(ate_rel_errs)):.4f}"
)

PEHE on IHDP datasets: 6.065 ± 0.8944
ATE relative error on IHDP datasets: 0.5718 ± 0.0953


## Lalonde Datasets (CPS and PSID)

In [22]:
import os
from typing import Callable, Tuple

current_dir = os.getcwd()


class RealCauseDataset(EvalDatasetCatalog, ABC):
    def __init__(
        self,
        name: str,
        csv_path_fn: Callable[[int], str],  # Returns the URL of the CSV file for the dataset, given an index
        n_tables: int = 100,
        test_ratio: float = 0.1,
        seed: int = 42,
        **kwargs,
    ):
        self.csv_path_fn = csv_path_fn
        self.test_ratio = test_ratio
        self.rngs = [np.random.default_rng(seed + i) for i in range(n_tables)]
        self.datasets = [self._get_data(i) for i in range(n_tables)]
        super().__init__(n_tables, name=name)

    def _get_data(self, idx: int) -> Tuple[CATE_Dataset, ATE_Dataset]:
        csv_path = self.csv_path_fn(idx)
        # Read the dataset
        data = pd.read_csv(csv_path)

        covariates_size = data.shape[1] - 5  # everything except for `t`, `y`, `y0`, `y1`, and `ite`
        # Define column names
        col_names = [f"x{i}" for i in range(1, covariates_size + 1)] + ["t", "y", "y0", "y1", "ite"]
        data.columns = col_names
        data = data.astype({"t": "bool"}, copy=False)

        # Convert to PyTorch tensors
        covariates = data.iloc[:, :covariates_size].values.astype(np.float32)  # Features
        treatments = data["t"].values.astype(np.float32)  # Treatment
        outcomes = data["y"].values.astype(np.float32)  # Factual outcomes

        cate = data["ite"].values.astype(np.float32)
        ate = cate.mean()

        # Split the dataset into train and test sets
        indices = self.rngs[idx].permutation(covariates.shape[0])
        split_idx = int(len(indices) * (1 - self.test_ratio))
        train_indices = indices[:split_idx]
        test_indices = indices[split_idx:]
        cate_dataset = CATE_Dataset(
            X_train=covariates[train_indices],
            t_train=treatments[train_indices],
            y_train=outcomes[train_indices],
            X_test=covariates[test_indices],
            true_cate=cate[test_indices],
        )
        ate_dataset = ATE_Dataset(
            X=covariates,
            t=treatments,
            y=outcomes,
            true_ate=float(ate),
        )
        return cate_dataset, ate_dataset

    def __getitem__(self, index) -> Tuple[CATE_Dataset, ATE_Dataset]:
        return self.datasets[index]


class RealCauseLalondePSIDDataset(RealCauseDataset):

    def __init__(self, **kwargs):
        LALONDE_PSID_CSV_PATH = lambda i: os.path.join(
            current_dir, f"data/realcause_datasets/lalonde_psid_sample{i}.csv"
        )
        super().__init__(name="LalondePSID", csv_path_fn=LALONDE_PSID_CSV_PATH, **kwargs)


class RealCauseLalondeCPSDataset(RealCauseDataset):

    def __init__(self, **kwargs):
        LALONDE_CPS_CSV_PATH = lambda i: os.path.join(current_dir, f"data/realcause_datasets/lalonde_cps_sample{i}.csv")
        super().__init__(name="LalondeCPS", csv_path_fn=LALONDE_CPS_CSV_PATH, **kwargs)

In [23]:
dataset = RealCauseLalondeCPSDataset()

pehes = []
ate_rel_errs = []

for i in range(len(dataset)):
    cate_dset, ate_dset = dataset[i]
    cate_pred = estimate_cate_dopfn(
        X_train=cate_dset.X_train,
        t_train=cate_dset.t_train,
        y_train=cate_dset.y_train,
        X_test=cate_dset.X_test,
    )
    pehe = np.sqrt(np.mean((cate_pred - cate_dset.true_cate) ** 2))
    pehes.append(pehe)

    ate_pred = estimate_ate_dopfn(
        X=ate_dset.X,
        t=ate_dset.t,
        y=ate_dset.y,
    )
    ate_rel_err = np.abs(ate_pred - ate_dset.true_ate) / np.abs(ate_dset.true_ate)
    ate_rel_errs.append(ate_rel_err)

print(f"PEHE on Lalonde CPS datasets: {np.mean(pehes):.4f} ± {np.std(pehes) / np.sqrt(len(pehes)):.4f}")
print(
    f"ATE relative error on Lalonde CPS datasets: {np.mean(ate_rel_errs):.4f} ± {np.std(ate_rel_errs) / np.sqrt(len(ate_rel_errs)):.4f}"
)

PEHE on Lalonde CPS datasets: 12014.6426 ± 31.8558
ATE relative error on Lalonde CPS datasets: 0.8744 ± 0.0060


In [26]:
dataset = RealCauseLalondePSIDDataset()

pehes = []
ate_rel_errs = []

for i in range(len(dataset)):
    cate_dset, ate_dset = dataset[i]
    cate_pred = estimate_cate_dopfn(
        X_train=cate_dset.X_train,
        t_train=cate_dset.t_train,
        y_train=cate_dset.y_train,
        X_test=cate_dset.X_test,
    )
    pehe = np.sqrt(np.mean((cate_pred - cate_dset.true_cate) ** 2))
    pehes.append(pehe)

    ate_pred = estimate_ate_dopfn(
        X=ate_dset.X,
        t=ate_dset.t,
        y=ate_dset.y,
    )
    ate_rel_err = np.abs(ate_pred - ate_dset.true_ate) / np.abs(ate_dset.true_ate)
    ate_rel_errs.append(ate_rel_err)

print(f"PEHE on Lalonde PSID datasets: {np.mean(pehes):.4f} ± {np.std(pehes) / np.sqrt(len(pehes)):.4f}")
print(
    f"ATE relative error on Lalonde PSID datasets: {np.mean(ate_rel_errs):.4f} ± {np.std(ate_rel_errs) / np.sqrt(len(ate_rel_errs)):.4f}"
)

PEHE on Lalonde PSID datasets: 20907.1973 ± 137.5429
ATE relative error on Lalonde PSID datasets: 0.9161 ± 0.0060
